In [ ]:
# maintenance done on 27/06/2022 @ 3:05 pm
# by Gabin

#@author: PristerM
#update by LLZ
#on 21/02/2022

from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os
import re
import requests
import tabula
import pdfplumber
import camelot

print("Running MY LFSA Web Scraping Tool v.1.0")


os.environ["PATH"] += r"C:\Program Files (x86)\gs\gs9.54.0\bin"
os.environ["PATH"] += r"C:\Program Files (x86)\gs\gs9.54.0\lib"

#Assigning current time, output file name and ExcelWriter object
now = datetime.datetime.now()
filename = 'MY LFSA SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

#Assigning the folders that are going to be used in the process
scriptfolder = os.path.dirname(os.path.abspath(__file__))
tempfolder = os.path.join(scriptfolder,'tempfolder')
os.chdir(scriptfolder)

#Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
	for temp_file in os.listdir(tempfolder):
		os.remove(os.path.join(tempfolder, temp_file))
else:
	os.mkdir(tempfolder)

#____Run the code for each of the following links____

regdict={'MY LFSA 1': 'https://www.labuanfsa.gov.my/areas-of-business/financial-services/banking/list-of-labuan-banks-and-investment-banks',
         'MY LFSA 2':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/banking/list-of-labuan-banks-and-investment-banks',
         'MY LFSA 3':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/insurance/list-of-labuan-insurance-insurance-related-entities',
         'MY LFSA 4':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/insurance/list-of-labuan-insurance-insurance-related-entities',
         'MY LFSA 5':'https://www.labuanfsa.gov.my/areas-of-business/labuan-service-providers/trust-companies-and-ancillary-services/list-of-labuan-trust-companies',
         'MY LFSA 6':'https://www.labuanfsa.gov.my/areas-of-business/labuan-service-providers/trust-companies-and-ancillary-services/list-of-labuan-trust-companies',
         'MY LFSA 7':'https://www.labuanibfc.com/areas-of-business/financial-services/leasing/list-of-leasing-companies',
         'MY LFSA 8':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/leasing/list-of-leasing-companies',
         'MY LFSA 9':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/commodity-trading/list-of-labuan-international-commodity-trading-companies',
         'MY LFSA 10':'https://www.labuanfsa.gov.my/areas-of-business/financial-services/capital-markets/list-of-fund-managers'}

print('The current folder is: {}\nThe temp folder is: {}'.format(scriptfolder, tempfolder))
chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory" : tempfolder, 
        "plugins.always_open_pdf_externally": True}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

try:
    os.mkdir(tempfolder)
except:
    prevfiles=os.listdir(tempfolder)
    os.chdir(tempfolder)
    for prf in prevfiles:
        os.remove(prf)
    print('The directory tempfolder already exists.')
os.chdir(tempfolder)##only if files are going to be downloaded here

processdate=now.strftime('%Y-%m-%d')
pattern = re.compile('([0-9]+)')

for reg in regdict:
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(3)
    
    if reg in ['MY LFSA 2', 'MY LFSA 4', 'MY LFSA 6']:
        
        soup=BeautifulSoup(driver.page_source,"html.parser")
        div = soup.find('div',{'class':'content list-container list-d-gap-large list-t-gap-medium list-m-gap-small list-d-col-1 list-t-col-1 list-m-col-1'})
        a_s = div.find_all('a')
        for a in a_s:
            if 'Surrendered & Revoked' in a.text.strip():
                print(a.text.strip())
                driver.get(a['href'])
                
                while len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:
                    print('Waiting for file to download')
                    sleep(2)
    
                print(os.listdir(tempfolder))
                pdf_file = os.listdir(tempfolder)[0]
                filePath = os.path.join(tempfolder, pdf_file)
    
                tables = camelot.read_pdf(filePath, pages='all', flavor='stream', edge_tol=50)
                # Iterate over the tables of each pages
                for i in range(tables.n):
                    df_Table = tables[i].df
        
                    for j in range(len(df_Table)):
                        if df_Table[0][j].strip() not in ['NO', ''] and len(pattern.findall(df_Table[0][j])) == 0:
                            regTyp = df_Table[0][j]
                        if df_Table[1][j].strip() not in ['','NAME']:
                            #print('Name: ', df_Table[1][j])
                            #print('regTyp: ', regTyp,'\n')
                            sqldict["Name"].append(df_Table[1][j])
                            sqldict["ListProcessDate"].append(processdate)
                            sqldict["RegCtry"].append("MY")    
                            sqldict["RegCode"].append("LFSA")
                            sqldict['ListCode'].append(reg.split(' ')[-1])
                            sqldict["Cntry"].append("MY")
                            sqldict["RegulationType"].append(regTyp)
                            
                            # Fill the rest with empty string
                            for key in sqldict.keys():
                                if len(sqldict['Name']) > len(sqldict[key]):
                                    sqldict[key].append('')
                                    
        os.remove(filePath)
                
    else:    
        
        soup=BeautifulSoup(driver.page_source,"html.parser")
        div=soup.find('div',{'class':'content list-container list-d-gap-none list-t-gap-none list-m-gap-none list-d-col-1 list-t-col-1 list-m-col-1'})
        
        if reg in ['MY LFSA 1', 'MY LFSA 3', 'MY LFSA 5', 'MY LFSA 9', 'MY LFSA 10']:
            anchors=div.find_all('a',{'data-so-type':'btn;1'},href=True)
            
        counter=0
    
        page_nav = soup.find('div',{'class':'page_navigation icon-size-m font-844CA48E-8825-412E-9881-0CBB0011A17F'})
        pages = len(soup.find_all('a',{'class': re.compile('^page_link*')}))

        print(pages)
    
        # When there are multiple pages to scrap and we need to click on next to access the next-page
        if pages != 0:
            for click_numb in range(1,pages+1):
                soup=BeautifulSoup(driver.page_source,"html.parser")
                div=soup.find('div',{'class':'content list-container list-d-gap-none list-t-gap-none list-m-gap-none list-d-col-1 list-t-col-1 list-m-col-1'})
                anchors=div.find_all('a',{'data-so-type':'btn;1'},href=True)
            
                if reg in ['MY LFSA 1', 'MY LFSA 3', 'MY LFSA 5', 'MY LFSA 9', 'MY LFSA 10']:
                    for a in anchors:
                        if "javascript" in a["href"]:
                            continue
        
                        counter+=1
                        driver.get('https://www.labuanfsa.gov.my/'+ a["href"])
                        sleep(1)
                        soup=BeautifulSoup(driver.page_source,"html.parser")
                        rows=soup.find_all('div',{'class':'a-inner-text'})
                        #print(len(rows))
                        #print(counter)
                        #print('Name: ',rows[0].text.strip())
                        sqldict["Name"].append(rows[0].text.strip())
                        sqldict["ListProcessDate"].append(processdate)
                        sqldict["RegCtry"].append("MY")    
                        sqldict["RegCode"].append("LFSA")
                        sqldict['ListCode'].append(reg.split(' ')[-1])
                        sqldict["Cntry"].append("MY")
                        sqldict["RegulationType"].append('Regulated')
                        
                        for row_index in range(len(rows)):
                            line_upper=rows[row_index].text.upper().strip()
                            if line_upper.startswith('ADDRESS') and len(sqldict["Name"])>len(sqldict["Address_1"]):
                                sqldict["Address_1"].append(rows[row_index+1].text.strip())
                            elif line_upper.startswith('MARKETING') and len(sqldict["Name"])>len(sqldict["Address_2"]):
                                if rows[row_index+1].text.strip() != 'N/A':
                                    sqldict["Address_2"].append(rows[row_index+1].text.strip())
                                else:
                                    sqldict["Address_2"].append('')
                            elif line_upper.startswith('TEL') and len(sqldict["Name"])>len(sqldict["Phone"]):
                                sqldict["Phone"].append(rows[row_index+1].text.strip())
                            elif line_upper.startswith('E-MAIL') and len(sqldict["Name"])>len(sqldict["Email"]):
                                sqldict["Email"].append(rows[row_index+1].text.strip())
                            elif line_upper.startswith('FAX') and len(sqldict["Name"])>len(sqldict["Fax"]):
                                sqldict["Fax"].append(rows[row_index+1].text.strip())
                    
                        # Fill the rest with empty string
                        for key in sqldict.keys():
                            if len(sqldict['Name']) > len(sqldict[key]):
                                sqldict[key].append('')
            
                # MY LFSA does not have anchors with href. It is a direct scrap of the name only.
                elif reg in ['MY LFSA 7', 'MY LFSA 8']:
                
                    counter+=1
                    soup=BeautifulSoup(driver.page_source,"html.parser")
                    div=soup.find('div',{'class':'content list-container list-d-gap-none list-t-gap-none list-m-gap-none list-d-col-1 list-t-col-1 list-m-col-1'})
                    rows=div.find_all('div',{'class':'a-inner-text'})
                
                    for cell in rows:
                        
                        name = cell.text.strip()
                        
                        if len(pattern.findall(name))> 0:
                            pass
                        else:
                            sqldict["Name"].append(name)
                            sqldict["ListProcessDate"].append(processdate)
                            sqldict["RegCtry"].append("MY")    
                            sqldict["RegCode"].append("LFSA")
                            sqldict['ListCode'].append(reg.split(' ')[-1])
                            sqldict["Cntry"].append("MY")
                            if reg == 'MY LFSA 7':
                                sqldict["RegulationType"].append('Regulated')
                            if reg == 'MY LFSA 8':
                                sqldict["RegulationType"].append('Ceased')
                        
                            # Fill the rest with empty string
                            for key in sqldict.keys():
                                if len(sqldict['Name']) > len(sqldict[key]):
                                    sqldict[key].append('')
                     
                #print('click_numb: ',click_numb)
                
                #Come back to the main page
                driver.get(regdict[reg])
                sleep(2)
            
                # click on next page click_numb's time
                if click_numb < pages:
                    for i in range(click_numb):
                        driver.find_element(By.CLASS_NAME, 'next_link').click()
                        sleep(3)
                    sleep(5)
                else:
                    pass
    
        # When there is only one page to scrap
        else:
            for a in anchors:
                if "javascript" in a["href"]:
                    continue
        
                counter+=1
                driver.get('https://www.labuanfsa.gov.my/'+ a["href"])
                sleep(1)
                soup=BeautifulSoup(driver.page_source,"html.parser")
                rows=soup.find_all('div',{'class':'a-inner-text'})
                #print(counter)
                #print('Name: ',rows[0].text.strip())
                
                sqldict["Name"].append(rows[0].text.strip())
                sqldict["ListProcessDate"].append(processdate)
                sqldict["RegCtry"].append("MY")    
                sqldict["RegCode"].append("LFSA")
                sqldict['ListCode'].append(reg.split(' ')[-1])
                sqldict["Cntry"].append("MY")
                
                for row_index in range(len(rows)):
                    line_upper=rows[row_index].text.upper().strip()
                    if line_upper.startswith('ADDRESS') and len(sqldict["Name"])>len(sqldict["Address_1"]):
                        sqldict["Address_1"].append(rows[row_index+1].text.strip())
                    elif line_upper.startswith('MARKETING') and len(sqldict["Name"])>len(sqldict["Address_2"]):
                        if rows[row_index+1].text.strip() != 'N/A':
                            sqldict["Address_2"].append(rows[row_index+1].text.strip())
                        else:
                            sqldict["Address_2"].append('')
                    elif line_upper.startswith('TEL') and len(sqldict["Name"])>len(sqldict["Phone"]):
                        sqldict["Phone"].append(rows[row_index+1].text.strip())
                    elif line_upper.startswith('E-MAIL') and len(sqldict["Name"])>len(sqldict["Email"]):
                        sqldict["Email"].append(rows[row_index+1].text.strip())
                    elif line_upper.startswith('FAX') and len(sqldict["Name"])>len(sqldict["Fax"]):
                        sqldict["Fax"].append(rows[row_index+1].text.strip())
                        
                # Fill the rest with empty string
                for key in sqldict.keys():
                    if len(sqldict['Name']) > len(sqldict[key]):
                        sqldict[key].append('')

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
#Moving the file to the output folder (this way it will be displayed in the Control Room)
sleep(3)

driver.quit()
    
    
    